# 🚀 VisionAI: Fine-Tuning YOLO11 for Hazardous Materials & Gas Cylinder Detection
### Semester V Cornerstone Project: *Design and Development of an Intelligent Object Detection using Deep Learning*

This notebook trains a custom **YOLO11** model on your uploaded dataset directly from **Google Drive** using a free **NVIDIA T4 GPU** in Google Colab.

---
### 📋 Instructions:
1. Go to **Runtime** > **Change runtime type** > Select **T4 GPU** > Click **Save**.
2. Run **Cell 1** to mount Google Drive and install Ultralytics.
3. Run **Cell 2** to unzip the dataset to Colab's fast SSD.
4. Run **Cell 3** to train for 30 epochs (~3 to 4 minutes).
5. Run **Cell 4** to save `best.pt` and training charts directly back to your Drive!

## Step 1: Mount Google Drive & Install Ultralytics

In [ ]:
# Mount your Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Install Ultralytics YOLO
!pip install -q ultralytics

import torch
print(f"\nPyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ GPU not enabled! Please go to Runtime > Change runtime type > T4 GPU")

## Step 2: Unzip Dataset from Drive to Fast Local SSD

In [ ]:
import os
import glob
import yaml
from pathlib import Path

# Look for the dataset zip file in your Drive
drive_zips = glob.glob('/content/drive/MyDrive/*.zip')
print("Found zip files in your Google Drive:")
for z in drive_zips:
    print(" - ", z)

# Auto-detect hazardous/materials/yolo zip or specify exact name
matched_zips = [z for z in drive_zips if any(k in os.path.basename(z).lower() for k in ['hazard', 'material', 'yolo', 'cylinder'])]
zip_to_extract = matched_zips[0] if matched_zips else (drive_zips[0] if drive_zips else None)

if zip_to_extract:
    print(f"\n📦 Extracting: {zip_to_extract} to /content/dataset/ ...")
    !mkdir -p /content/dataset
    !unzip -q "{zip_to_extract}" -d /content/dataset/
    print("✅ Extraction Complete!")
else:
    print("⚠️ No zip file found in MyDrive root! Please check the zip filename in your Google Drive.")

# Fix and verify data.yaml
yaml_path = "/content/dataset/data.yaml"
if os.path.exists(yaml_path):
    with open(yaml_path, 'r') as f:
        data_cfg = yaml.safe_load(f)
    
    data_cfg['train'] = '/content/dataset/train/images'
    data_cfg['val'] = '/content/dataset/valid/images'
    if 'test' in data_cfg:
        data_cfg['test'] = '/content/dataset/test/images'
        
    with open(yaml_path, 'w') as f:
        yaml.dump(data_cfg, f)
        
    print("\n--- Target Classes in Dataset ---")
    for idx, name in enumerate(data_cfg.get('names', [])):
        print(f"  Class {idx}: {name}")

## Step 3: Train YOLO11 Model via Transfer Learning
We fine-tune the YOLO11 model on the hazardous objects dataset on GPU for 30 epochs.

In [ ]:
from ultralytics import YOLO

# Load pre-trained YOLO11 Nano
model = YOLO("yolo11n.pt")

# Train on GPU
results = model.train(
    data="/content/dataset/data.yaml",
    epochs=30,
    imgsz=640,
    batch=16,
    device=0 if torch.cuda.is_available() else 'cpu',
    save=True,
    plots=True,
    name="hazard_specialist_yolo11",
    verbose=True
)

print("\n🎉 Fine-Tuning Completed Successfully!")

## Step 4: Validate Model & Save Weights Directly to Google Drive
Saves `best.pt`, `results.png`, and `confusion_matrix.png` directly back to your Google Drive root and initiates local download.

In [ ]:
import shutil
from pathlib import Path
from google.colab import files
from IPython.display import Image, display

# 1. Print validation metrics
metrics = model.val()
print(f"\n🏆 Validation mAP@50: {metrics.box.map50:.2%}")
print(f"🏆 Validation mAP@50-95: {metrics.box.map:.2%}")

run_dir = Path(results.save_dir)
best_weights = run_dir / "weights" / "best.pt"

# 2. Save directly to your Google Drive
if best_weights.exists():
    shutil.copy(best_weights, "/content/drive/MyDrive/best_hazard_model.pt")
    print("\n💾 Saved 'best_hazard_model.pt' directly to your Google Drive!")

for chart in ["results.png", "confusion_matrix.png", "PR_curve.png"]:
    if (run_dir / chart).exists():
        shutil.copy(run_dir / chart, f"/content/drive/MyDrive/{chart}")

# 3. Display evaluation graphs in notebook
if (run_dir / "results.png").exists():
    display(Image(str(run_dir / "results.png")))

# 4. Trigger direct browser download
print("\n⬇️ Downloading best.pt to your computer...")
files.download(str(best_weights))